# 04 — Land Surface Preprocessing

Inventories and validates DEM, Slope, Aspect, NDVI, LST Day and
Distance-to-Sea predictor files.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import rasterio

predictor_root = RAW_DIR / "predictors"
raster_files = sorted([
    *predictor_root.rglob("*.tif"),
    *predictor_root.rglob("*.tiff"),
])

if not raster_files:
    raise FileNotFoundError("No predictor GeoTIFF files were found.")

records = []
for path in raster_files:
    with rasterio.open(path) as src:
        records.append({
            "predictor": path.parent.name,
            "file": path.name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution_x": src.res[0],
            "resolution_y": src.res[1],
            "nodata": src.nodata,
            "dtype": src.dtypes[0],
        })

predictor_inventory = pd.DataFrame(records)
display(predictor_inventory)

In [ ]:
output_dir = INTERIM_DIR / "inventories"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "predictor_raster_inventory.csv"
predictor_inventory.to_csv(output_path, index=False)
print(f"Saved: {output_path}")